In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Gold Circuits
# MAGIC
# MAGIC Read the Silver `circuits` data, select the required business
# MAGIC columns, create a business-friendly `location` column,
# MAGIC and write the result to the Gold layer.

# COMMAND ----------

# MAGIC %run "../00.common/01.environment-config.py"

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

# ============================================================
# GOLD SOURCE AND TARGET PATHS
# ============================================================

silver_path = f"{s3_root_path}/Silver/Nisarga/circuits"

gold_path = f"{s3_root_path}/Gold/Nisarga/circuits"

print("Silver Path :", silver_path)
print("Gold Path   :", gold_path)

# COMMAND ----------

# ============================================================
# STEP 1 - READ SILVER CIRCUITS DATA
# ============================================================

silver_circuits_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(silver_circuits_df)

# COMMAND ----------

# ============================================================
# STEP 2 - SELECT GOLD COLUMNS
# ============================================================

circuits_gold_df = (
    silver_circuits_df
    .select(
        "circuit_id",
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country"
    )
)

display(circuits_gold_df)

# COMMAND ----------

# ============================================================
# STEP 3 - ADD BUSINESS-FRIENDLY DERIVED COLUMN
# ============================================================

circuits_gold_df = (
    circuits_gold_df
    .withColumn(
        "location",
        F.concat_ws(
            ", ",
            F.col("locality"),
            F.col("country")
        )
    )
)

display(circuits_gold_df)

# COMMAND ----------

# ============================================================
# STEP 4 - VALIDATE GOLD DATA
# ============================================================

print("Gold Records :", circuits_gold_df.count())

print(
    "Distinct Circuits :",
    circuits_gold_df
    .select("circuit_id")
    .distinct()
    .count()
)

print(
    "Null Circuit IDs :",
    circuits_gold_df
    .filter(F.col("circuit_id").isNull())
    .count()
)

# COMMAND ----------

# ============================================================
# STEP 5 - WRITE GOLD DATA
# ============================================================

(
    circuits_gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)

print("Gold Circuits data successfully written to:")
print(gold_path)

# COMMAND ----------

# ============================================================
# STEP 6 - READ AND VALIDATE GOLD DATA
# ============================================================

gold_circuits_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

display(gold_circuits_df)

print("Silver Records :", silver_circuits_df.count())
print("Gold Records   :", gold_circuits_df.count())

# COMMAND ----------

# ============================================================
# STEP 7 - DISPLAY GOLD SCHEMA
# ============================================================

gold_circuits_df.printSchema()